[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/00_orientation/00A_tour_of_the_pipeline.ipynb)

# Tour of the Berkeley Housing Pipeline

**Welcome!** This notebook gives you a plain-language introduction to:
1. What a housing permit pipeline is
2. How Berkeley's data sources work
3. How this project's notebooks fit together

No coding experience required to read this. We'll look at one small code example at the end.

---

## What is a Housing Permit Pipeline?

When someone wants to build housing in a city, they don't just start construction. They go through a **permit pipeline**:

```
PROPOSAL → REVIEW → APPROVAL → PERMIT → CONSTRUCTION → OCCUPANCY
   |          |         |          |           |            |
   |          |         |          |           |            └── People move in!
   |          |         |          |           └── Building goes up
   |          |         |          └── City issues building permit
   |          |         └── City approves the project
   |          └── City reviews plans, environmental impact, etc.
   └── Developer submits application
```

### Why does this matter?

- **For residents:** Understanding what's being built in your neighborhood
- **For cities:** Tracking progress toward housing goals (California requires cities to report annually)
- **For researchers:** Studying why housing takes so long to build
- **For advocates:** Identifying bottlenecks and delays

### The California Connection

California requires every city to submit an **Annual Progress Report (APR)** to the Department of Housing and Community Development (HCD). This report tracks:
- How many units were permitted
- How many were built
- Progress toward the city's housing allocation (RHNA)

This project helps make that tracking **transparent and reproducible**.

## Berkeley's Data Sources

To track housing in Berkeley, we pull data from several places:

| Data Source | What It Contains | How We Use It |
|-------------|------------------|---------------|
| **Permits Portal** | Zoning & building permits | Track applications, approvals, issuance |
| **Assessor Parcels** | Property boundaries, APNs | Unique identifier for each property |
| **Zoning Data** | What can be built where | Understand constraints |
| **Inspections** | Construction progress | Know when buildings are done |
| **BuildingEye** | Design review details | Get timeline data |

### The Challenge

This data is scattered across different systems, uses different formats, and has inconsistencies. For example:
- The same address might appear as "123 Main St" or "123 MAIN STREET"
- Status codes vary: "Approved" vs "APPRVD" vs "Pending Final Action"
- Some records lack key dates

**This project cleans, standardizes, and connects all this data.**

## How the Notebook Series Fit Together

We've organized the work into **five series**, each in its own folder:

### A-Series: Data Collection (`01_collection/`)
**What:** Get data from various sources and clean it up

| Notebook | Purpose |
|----------|--------|
| A1 | Connect to Berkeley Open Data portal |
| A2 | Standardize messy addresses |
| A3 | Geocode addresses (find lat/lon) |
| A4 | Match projects to Assessor Parcel Numbers |
| A5 | Import timeline data from BuildingEye |

---

### B-Series: Timeline Tracking (`02_tracking/`)
**What:** Track where each project is in the pipeline

| Notebook | Purpose |
|----------|--------|
| B1 | Track lifecycle: proposal → occupancy |
| B2 | Classify status codes into standard categories |
| B3 | Identify stalled projects (>180 days inactive) |

---

### C-Series: Analysis (`03_analysis/`)
**What:** Analyze patterns and trends

| Notebook | Purpose |
|----------|--------|
| C1 | Pipeline analysis: how many units at each stage |
| C2 | Timeline analysis: where are the bottlenecks |
| C3 | Proposal vs reality: what got built vs approved |

---

### D-Series: Reporting (`04_reporting/`)
**What:** Generate reports and dashboards

| Notebook | Purpose |
|----------|--------|
| D1 | Monthly report generator |
| D2 | Dashboard data export (Datasette) |
| D3 | Alerts and monitoring |

---

### F-Series: Feasibility (`05_feasibility/`)
**What:** Understand development economics

| Notebook | Purpose |
|----------|--------|
| F1 | Basic development math: costs, revenue, returns |
| F2 | Full pro forma analysis with policy scenarios |

---

### The Flow

```
A-Series (collect) → B-Series (track) → C-Series (analyze) → D-Series (report)
                                                              ↓
                                              F-Series (understand economics)
```

## Let's Look at Some Real Data

Now let's see what this data actually looks like. We'll load the main housing projects table and explore a few rows.

In [ ]:
# First, let's import the tools we need
# pandas is a Python library for working with data tables
import pandas as pd
from pathlib import Path

# Find our project's data folder
# We look for the housing_projects_FINAL.csv file
data_paths = [
    Path('data/processed/housing_projects_FINAL.csv'),           # If running from root
    Path('../data/processed/housing_projects_FINAL.csv'),        # If running from 00_orientation
    Path('/content/berkeley-housing-analysis/data/processed/housing_projects_FINAL.csv')  # Colab
]

# Try each path until we find the file
df = None
for path in data_paths:
    if path.exists():
        df = pd.read_csv(path)
        print(f"Loaded data from: {path}")
        break

if df is None:
    print("Data file not found. If running in Colab, clone the repo first.")

In [ ]:
# Let's see what columns we have
if df is not None:
    print(f"We have {len(df)} housing projects in our database.")
    print(f"\nEach project has {len(df.columns)} data fields:")
    print(df.columns.tolist())

In [ ]:
# Let's look at a few key columns for the first 5 projects
if df is not None:
    # Select just the most important columns
    key_columns = ['address_display', 'net_units', 'status', 'year']
    
    # Filter to columns that exist
    available_columns = [c for c in key_columns if c in df.columns]
    
    print("Sample of 5 housing projects:")
    print("="*60)
    display(df[available_columns].head())
    
    print("\nWhat these columns mean:")
    print("  address_display: The street address of the project")
    print("  net_units: How many new housing units (homes/apartments)")
    print("  status: Where the project is in the pipeline")
    print("  year: When the project was filed or approved")

In [ ]:
# Quick summary: how many units total?
if df is not None and 'net_units' in df.columns:
    total_units = df['net_units'].sum()
    avg_units = df['net_units'].mean()
    
    print(f"Berkeley Housing Pipeline Summary")
    print(f"="*40)
    print(f"Total projects tracked: {len(df)}")
    print(f"Total housing units: {total_units:,.0f}")
    print(f"Average units per project: {avg_units:.1f}")

## What's Next?

Now that you understand the big picture, here are your options:

### If you have 30 minutes:
1. **Run `00B_first_notebook_in_colab.ipynb`** - Create your first chart
2. Come back here and re-read the series descriptions above

### If you want to dive deeper:
1. **Start with `A1_data_sources_setup.ipynb`** in `01_collection/` - See how we get data
2. **Then try `C1_pipeline_analysis.ipynb`** in `03_analysis/` - Analyze the pipeline

### If you're interested in economics:
1. **Jump to `F1_development_math.ipynb`** in `05_feasibility/` - Learn why housing is expensive

---

## Key Takeaways

1. **Housing goes through a pipeline** from proposal to occupancy
2. **Data is scattered** across multiple city systems
3. **This project connects it all** into one coherent picture
4. **California requires cities to report** on housing progress (APR)
5. **You can learn data science** while studying a real civic issue

Welcome to civic data science!